# 03 - RLHF Eğitimi: Ödül Modeli + PPO (Çoklu-LoRA-Adaptör Yöntemi)

**Amaç:** `Model_SFT` üzerinden başlayarak,

1. `Anthropic/hh-rlhf` (Bai vd., 2022) tercih verisiyle bir **Ödül Modeli**
   (Reward Model) LoRA adaptörü eğitmek (`trl.RewardTrainer`),
2. Bu ödül adaptörünü kullanarak **PPO** (Proximal Policy Optimization) ile
   `Model_RLHF`'yi elde etmek (`trl.PPOTrainer`).

**Bellek optimizasyonu:** Dört ayrı model (policy, reference, reward, value)
belleğe yüklemek yerine, TRL'nin **çoklu-adaptör (multi-adapter) RL** yöntemi
kullanılır: `trl.AutoModelForCausalLMWithValueHead`, tek bir taban model
üzerinde **PPO-politika adaptörü** ile **ödül-modeli adaptörü** arasında
`set_adapter` ile dinamik geçiş yapar; referans log-olasılıkları da PPO
adaptörü geçici olarak devre dışı bırakılarak (adapter disable) aynı taban
model üzerinden hesaplanır (Hu vd., 2022; Dettmers vd., 2023).

> **Alternatif:** Üretim ölçeğinde (ör. Qwen2.5-7B) PPO yerine Shao vd. (2024)
> tarafından önerilen **GRPOTrainer** (`trl.GRPOTrainer`, daha yeni TRL
> sürümlerinde mevcuttur) değer (value) ağı gerektirmediği için tercih
> edilebilir; bu defterde API kararlılığı nedeniyle klasik `PPOTrainer`
> kullanılmıştır.

> Bu defter Google Colab Pro (A100/L4 GPU) üzerinde çalıştırılmak üzere tasarlanmıştır.

## 0. Ortam Kurulumu

In [ ]:
# --- Google Colab ortam kurulumu -------------------------------------------
# Bu hucre, projenin tamamini (Datasets/, src/, notebooks/, outputs/) iceren
# "Uygulama" klasorunu Google Drive'a yukledikten sonra calistirilmalidir.
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # TODO: Kendi Google Drive yolunuza gore guncelleyin
    PROJECT_ROOT = "/content/drive/MyDrive/Uygulama"
else:
    PROJECT_ROOT = os.path.abspath("..")

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IN_COLAB:", IN_COLAB)

In [ ]:
# NOT: Surumler, TRL'nin klasik `PPOTrainer` + `AutoModelForCausalLMWithValueHead`
# coklu-adaptor (multi-adapter) RL ozelligiyle dogrulanmis, bilinen-uyumlu bir
# kombinasyon olacak sekilde SABITLENMISTIR (API kararliligi icin).
%pip install -q -U "transformers==4.44.2" "trl==0.9.6" "peft==0.11.1" "bitsandbytes==0.43.1" "accelerate==0.32.1" "datasets==2.20.0" "numpy==1.26.4" sentencepiece

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import alignment_pipeline as ap

ap.set_global_seed(42)
print("Cihaz:", ap.get_device())

In [ ]:
config = ap.ExperimentConfig(
    dataset_root=os.path.join(PROJECT_ROOT, "Datasets"),
    output_root=os.path.join(PROJECT_ROOT, "outputs"),
)
ap.ensure_dir(config.output_root)
print("Taban model:", config.base_model_id)
print("Veri kok dizini:", config.dataset_root)
print("Cikti kok dizini:", config.output_root)

In [ ]:
SFT_MERGED_DIR = config.path("sft", "merged_model")  # 02_SFT_Training.ipynb çıktısı (Model_SFT)
assert os.path.isdir(SFT_MERGED_DIR), "Önce 02_SFT_Training.ipynb çalıştırılmalı (Model_SFT bulunamadı)."

RM_DIR = config.path("reward_model")
RM_ADAPTER_DIR = os.path.join(RM_DIR, "adapter")
RM_LOGS_DIR = os.path.join(RM_DIR, "logs")
PPO_DIR = config.path("ppo")
for d in (RM_ADAPTER_DIR, RM_LOGS_DIR, PPO_DIR):
    ap.ensure_dir(d)

QUANT_BITS = 4

## 1. Adım A - Ödül Modeli (Reward Model) Eğitimi

`RewardTrainer`, `chosen`/`rejected` ikili (pairwise) tercihlerden,
`AutoModelForSequenceClassification` (num_labels=1) üzerine kurulu bir LoRA
(SEQ_CLS) adaptörü eğitir. Bu adaptör daha sonra PPO aşamasında
`reward_adapter` olarak aynı taban modele takılacaktır.

In [ ]:
hh = ap.load_hh_rlhf_preference(config.dataset_root, subsets=config.hh_rlhf_subsets)
rm_dataset = ap.build_reward_model_dataset(hh)
print({k: len(v) for k, v in rm_dataset.items()})

In [ ]:
rm_tokenizer = ap.load_tokenizer(SFT_MERGED_DIR)
rm_lora_config = ap.make_lora_config(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=ap.QWEN2_LORA_TARGET_MODULES,
    task_type="SEQ_CLS",
)

rm_trainer, rm_model = ap.run_reward_model(
    model_id=SFT_MERGED_DIR,
    train_dataset=rm_dataset["train"],
    eval_dataset=rm_dataset["test"],
    tokenizer=rm_tokenizer,
    lora_config=rm_lora_config,
    output_dir=RM_DIR,
    max_length=1024,
    num_train_epochs=1.0,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1.41e-5,
    quant_bits=QUANT_BITS,
)

In [ ]:
rm_log_df = ap.trainer_log_history_to_df(rm_trainer)
ap.save_json(rm_trainer.state.log_history, os.path.join(RM_LOGS_DIR, "reward_log_history.json"))
ap.plot_loss_curve(rm_log_df, loss_col="loss", title="Ödül Modeli Eğitim Kaybı")

In [ ]:
rm_metrics = ap.evaluate_reward_accuracy(rm_trainer)
print("Ödül Modeli değerlendirme metrikleri (doğruluk dahil):", rm_metrics)
ap.save_json(rm_metrics, os.path.join(RM_LOGS_DIR, "reward_eval_metrics.json"))

In [ ]:
rm_model.save_pretrained(RM_ADAPTER_DIR)
rm_tokenizer.save_pretrained(RM_ADAPTER_DIR)
print("Ödül adaptörü kaydedildi ->", RM_ADAPTER_DIR)

## 2. Adım B - PPO (Çoklu-Adaptör RLHF)

Burada **tek bir** `AutoModelForCausalLMWithValueHead` nesnesi oluşturulur:
- `peft_config` → PPO politika adaptörü (fresh, Model_SFT üzerine eklenir)
- `reward_adapter` → yukarıda eğitilen ödül adaptörü

Böylece ayrı bir referans model veya ayrı bir ödül modeli belleğe yüklenmez.

In [ ]:
ppo_lora_config = ap.make_lora_config(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=ap.QWEN2_LORA_TARGET_MODULES,
    task_type="CAUSAL_LM",
)

ppo_model = ap.build_ppo_value_head_model(
    base_model_id=SFT_MERGED_DIR,
    ppo_lora_config=ppo_lora_config,
    reward_adapter_path=RM_ADAPTER_DIR,
    quant_bits=QUANT_BITS,
)
ap.print_gpu_memory_summary("PPO modeli yuklendikten sonra (tek taban model + 2 LoRA adaptörü)")

In [ ]:
ppo_tokenizer = ap.load_tokenizer(SFT_MERGED_DIR)
prompt_dataset = ap.build_ppo_prompt_dataset(hh, ppo_tokenizer, max_prompt_tokens=config.max_prompt_length, split="train")
print("PPO prompt veri kümesi boyutu:", len(prompt_dataset))

> **Not (hızlı deneme):** PoC aşamasında `num_ppo_steps` değerini küçük tutup
> (ör. 20-50) hızlı doğrulama yapabilir, ardından tam deney için artırabilirsiniz.

In [ ]:
ppo_trainer, ppo_history = ap.run_ppo_training(
    model=ppo_model,
    tokenizer=ppo_tokenizer,
    prompt_dataset=prompt_dataset,
    output_dir=PPO_DIR,
    num_ppo_steps=200,
    batch_size=16,
    mini_batch_size=4,
    ppo_epochs=4,
    learning_rate=1.4e-5,
    init_kl_coef=0.2,
    target_kl=6.0,
    max_new_tokens=128,
)
ppo_history.tail()

## 3. PPO Eğitim Eğrileri (Nicel)

In [ ]:
ap.plot_ppo_history(ppo_history, save_path_prefix=os.path.join(PPO_DIR, "logs_"))

## 4. Sonuç

- Ödül adaptörü: `outputs/reward_model/adapter`
- PPO (politika) adaptörü ve eğitim geçmişi: `outputs/ppo/ppo_adapter`,
  `outputs/ppo/ppo_training_history.csv` (bu dosyalar `run_ppo_training`
  tarafından otomatik kaydedilmiştir)

`Model_RLHF` = Model_SFT (taban) + `ppo_adapter` (LoRA). Sıradaki adım:
`04_DPO_Training.ipynb`.